# U-Net Segmentation Toolbox for Jupyter Notebook

Click a button to run **Train**, **Test**, or **Predict** mode.  
All settings are automatically loaded from `dataset_path.yaml`.

In [1]:
import os, sys, subprocess, threading
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Paths ---
UNET_DIR = os.path.abspath('.')
if not os.path.exists(os.path.join(UNET_DIR, 'unet_segmentation_sigmoid_aug.py')):
    UNET_DIR = os.path.abspath('unet')
SCRIPT = os.path.join(UNET_DIR, 'unet_segmentation_sigmoid_aug.py')
CONFIG = os.path.join(UNET_DIR, 'dataset_path.yaml')

print(f'Script : {SCRIPT}')
print(f'Config : {CONFIG}')

Script : e:\_USC\git\urban-tree-canopy\unet\unet_segmentation_sigmoid_aug.py
Config : e:\_USC\git\urban-tree-canopy\unet\dataset_path.yaml


In [2]:
# --- Runner ---
log = widgets.Output(layout=widgets.Layout(
    border='1px solid #aaa', min_height='250px', max_height='600px',
    overflow_y='auto', width='100%'
))
status_label = widgets.HTML(value='<b>Status:</b> Ready')
_proc = None


def run(mode):
    """Run: python unet_segmentation_sigmoid_aug.py <mode> --config dataset_path.yaml"""
    global _proc
    cmd = [sys.executable, '-u', SCRIPT, mode, '--config', CONFIG]
    mode_names = {'train': 'TRAINING', 'test': 'TESTING', 'predict': 'PREDICTING'}
    mode_label = mode_names.get(mode, mode.upper())

    status_label.value = f'<b>Status:</b> <span style="color:green">{mode_label} in progress...</span>'

    log.clear_output(wait=True)
    with log:
        print(f'>>> {mode_label}')
        print(f'Command: {" ".join(cmd)}')
        print('=' * 70)

    def _stream():
        global _proc
        try:
            _proc = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                bufsize=1, universal_newlines=True, cwd=UNET_DIR
            )
            for line in iter(_proc.stdout.readline, ''):
                if line:
                    log.append_stdout(line)
            _proc.stdout.close()
            _proc.wait()
            rc = _proc.returncode
            log.append_stdout('\n' + '=' * 70 + '\n')
            log.append_stdout(f'Done (exit code {rc})\n')
            if rc == 0:
                status_label.value = f'<b>Status:</b> <span style="color:blue">{mode_label} completed ✅</span>'
            else:
                status_label.value = f'<b>Status:</b> <span style="color:red">{mode_label} failed ❌ (code {rc})</span>'
        except Exception as e:
            log.append_stdout(f'\nERROR: {e}\n')
            status_label.value = f'<b>Status:</b> <span style="color:red">Error: {e}</span>'
        finally:
            _proc = None

    threading.Thread(target=_stream, daemon=True).start()


def on_stop(b):
    global _proc
    if _proc:
        _proc.kill(); _proc = None
        status_label.value = '<b>Status:</b> <span style="color:orange">Stopped ⛔</span>'
        log.append_stdout('\n--- Stopped by user ---\n')


# --- Buttons ---
bw = widgets.Layout(width='130px', height='38px')
btn_t = widgets.Button(description='Train',   button_style='success', layout=bw)
btn_e = widgets.Button(description='Test',    button_style='primary', layout=bw)
btn_p = widgets.Button(description='Predict', button_style='warning', layout=bw)
btn_s = widgets.Button(description='Stop',    button_style='danger',  layout=bw)
btn_t.on_click(lambda b: run('train'))
btn_e.on_click(lambda b: run('test'))
btn_p.on_click(lambda b: run('predict'))
btn_s.on_click(on_stop)

display(widgets.HBox([btn_t, btn_e, btn_p, btn_s], layout=widgets.Layout(gap='8px')))
display(status_label)
display(log)

HTML(value='<b>Status:</b> Ready')

Output(layout=Layout(border_bottom='1px solid #aaa', border_left='1px solid #aaa', border_right='1px solid #aa…

---
To change settings, edit `unet/dataset_path.yaml` and **re-run this notebook**.  
All paths and hyperparameters are automatically loaded from the YAML file.